# 03 — Review Clustering / Truth Discovery

Self-contained notebook: no `%run` dependency on project scripts.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import yaml

# In Colab, change only this path after mounting Google Drive.
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "src" else Path(".").resolve()
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

K-Means discovers patterns only. A cluster must not be called fake without external ground truth. Human mapping is performed after inspecting the profile.

In [ ]:
import joblib
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

reviews = pd.read_csv(PROCESSED / "reviews_clean.csv")
cluster_features = ["Comment_Length", "rating", "Sentiment_Score", "has_image"]
n_clusters = int(CONFIG["clustering"]["n_clusters"])
if len(reviews) < n_clusters:
    raise ValueError(f"Need at least {n_clusters} reviews; found {len(reviews)}")

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KMeans(n_clusters=n_clusters, random_state=CONFIG["project"]["random_state"], n_init=10)),
])
reviews["Cluster_Label"] = pipeline.fit_predict(reviews[cluster_features].fillna(0))
joblib.dump(pipeline, OUTPUT / "models" / "kmeans_reviews.joblib")

In [ ]:
cluster_profile = reviews.groupby("Cluster_Label").agg(
    Review_Count=("review_id", "count"),
    Avg_Comment_Length=("Comment_Length", "mean"),
    Avg_Rating=("rating", "mean"),
    Image_Rate=("has_image", "mean"),
    Avg_Sentiment_Score=("Sentiment_Score", "mean"),
).reset_index()
reviews.to_csv(PROCESSED / "reviews_clustered.csv", index=False)
cluster_profile.to_csv(OUTPUT / "reports" / "cluster_profile.csv", index=False)
display(cluster_profile)

In [ ]:
# Enter only a mapping confirmed by the team after examining cluster_profile.
mapping = {int(key): float(value) for key, value in CONFIG["clustering"].get("authenticity_mapping", {}).items()}
if not mapping:
    print("BLOCKED: add the human-confirmed mapping to config.yaml; no authenticity scores were fabricated.")
else:
    reviews["Is_Authentic"] = reviews["Cluster_Label"].map(mapping)
    if reviews["Is_Authentic"].isna().any():
        raise ValueError("Mapping must cover every cluster")
    shop_authenticity = reviews.groupby("Shop_ID").agg(Total_Reviews=("review_id", "count"), Authentic_Review_Rate=("Is_Authentic", "mean")).reset_index()
    shop_authenticity.to_csv(OUTPUT / "reports" / "Shop_Authenticity_Report.csv", index=False)
    display(shop_authenticity)